# Experiment 3 — MultipleNegativesRankingLoss (fully on Colab)

**One controlled independent variable:** the training objective, TripletLoss → **MultipleNegativesRankingLoss (MNRL)**.

Everything else is held identical to Experiments 1–2: dataset, question-level split, source triplets, seed, encoder (all-MiniLM-L6-v2), learning rate, optimizer, scheduler, epochs, checkpoint cadence (every 50 steps, all retained), validation-first checkpoint selection (val MRR@10, tie-break HitRate@10), metrics, and evaluation protocol.

**Entailments of the IV (not separate variables):**
- Triplets → (anchor, positive) pairs (the negative is discarded; MNRL builds negatives in-batch). All rows kept (each pair recurs ~3×) so the step budget and checkpoint steps match Experiment 2.
- Batch sampler = `NO_DUPLICATES` (the officially recommended MNRL setup; avoids a pair's duplicate becoming a self-false-negative).
- In-batch false negatives are left **unmasked** (faithful to standard MNRL) and documented as a limitation.

**Only edit `REPO_URL`, set Runtime → GPU, then Runtime → Run all.** No uploads, no local steps.

In [ ]:
# --- The ONLY cell you edit ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- set this

In [ ]:
# --- Verify GPU (Runtime > Change runtime type > GPU) ---
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# --- Clone repo (includes datasets/ and reference/) and install pinned deps ---
import os
if not os.path.exists("distractor"):
    !git clone {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt
assert os.path.exists("datasets/train.csv"), "datasets/train.csv missing from repo"
print("Raw dataset present:", os.listdir("datasets"))

In [ ]:
# --- Auto-generate splits if missing (Stage 01) ---
import os
if not os.path.exists("outputs/results/train_qdp.csv"):
    !python scripts/01_prepare_dataset.py
else:
    print("Splits already present — skipping Stage 01.")

In [ ]:
# --- Auto-generate triplets if missing (Stage 03) ---
import os
if not os.path.exists("outputs/triplets/train_triplets.jsonl"):
    !python scripts/03_create_triplets.py
else:
    print("Triplets already present — skipping Stage 03.")

In [ ]:
# --- Materialize (anchor, positive) pairs if missing (transparency artifact) ---
# Training also derives these internally from the same triplets via
# triplets_to_pairs(), so this file is identical to what MNRL consumes; it is
# written only so the pairs can be inspected.
import os, json, sys
sys.path.insert(0, ".")
from src.triplet_construction import load_triplets, triplets_to_pairs
os.makedirs("outputs/triplets", exist_ok=True)
if not os.path.exists("outputs/triplets/train_pairs.jsonl"):
    pairs = triplets_to_pairs(load_triplets("train_triplets.jsonl"))
    with open("outputs/triplets/train_pairs.jsonl", "w") as f:
        for a, p in pairs:
            f.write(json.dumps({"anchor": a, "positive": p}) + "\n")
    print(f"Wrote {len(pairs):,} (anchor, positive) pairs")
else:
    print("Pairs already present.")

In [ ]:
# --- Train: MNRL, full checkpoint retention (only the objective differs from Exp 2) ---
!python scripts/train_gpu.py --objective mnrl \
  --save-steps 50 --eval-steps 50 --keep-all-checkpoints \
  --output outputs/models/finetuned_mnrl

In [ ]:
# --- Sanity check: all checkpoints retained ---
import os
ck = sorted(c for c in os.listdir("outputs/models/finetuned_mnrl/checkpoints") if c.startswith("checkpoint-"))
print(f"{len(ck)} checkpoints kept:", ck)

In [ ]:
# --- Evaluate every checkpoint (validation-first; test touched once) ---
# Identity-vs-Exp1 loss check is skipped: the objective changed by design, so
# the training-loss trajectory is deliberately not comparable. The Exp 2
# retrieval trajectory is loaded for comparison instead.
!python scripts/06_evaluate_checkpoints.py \
  --model-dir outputs/models/finetuned_mnrl \
  --experiment-name "Experiment 3" \
  --skip-identity-check \
  --compare-val-csv reference/exp2_checkpoint_metrics_val.csv \
  --fig-dir-name exp3_trajectory

In [ ]:
# --- Show the auto-generated report inline ---
print(open("outputs/results/checkpoint_trajectory_report.md").read())

In [ ]:
# --- Package all outputs and download ---
!zip -qr exp3_results.zip \
  outputs/results/checkpoint_metrics_val.csv \
  outputs/results/final_test_results.csv \
  outputs/results/checkpoint_trajectory_report.md \
  outputs/figures/exp3_trajectory \
  outputs/models/finetuned_mnrl/training_log.json
from google.colab import files
files.download("exp3_results.zip")